In [3]:
import pandas as pd
import math
import pandas as pd
import numpy as np
import statistics
from statistics import mean
import json
import pyserini
import pickle

In [4]:
from pyserini.index.lucene import IndexReader

# Initialize from a pre-built index:
index_reader = IndexReader.from_prebuilt_index('msmarco-v1-passage')

# Initialize from an index path:
# index_reader = IndexReader('indexes/index-robust04-20191213/')

In [6]:
collection = pd.read_csv("collection.tsv" , sep='\t' , names=['docid','passage'])

In [41]:
hard_queries = pd.read_csv("topics.tsv" , sep='\t' , names=['id' , 'query'])
# hard_queries

In [9]:
DL_2019_queries = pd.read_csv("DL_2019/queries.trec-dl-2019.judged.tsv" , sep='\t' , names=['id' , 'query'])
# print(DL_2019_queries)

In [42]:
bm25scores = pd.read_csv("bm25.run" , sep=' ', names=['qid','C2','docid','C4','score','C6'])
# bm25scores

In [44]:
bm25scores = bm25scores[['qid', 'docid', 'score']]

bm25scores['qid'] = bm25scores['qid'].astype(int)
bm25scores['docid'] = bm25scores['docid'].astype(int)
bm25scores['score'] = bm25scores['score'].astype(float)
bm25scores

,qid,docid,score
0,915593,82107,25.859501
1,915593,1772930,25.299900
2,915593,6923052,25.073299
3,915593,8178998,23.998600
4,915593,3523599,23.721701
...,...,...,...
49995,273695,6485471,10.901796
49996,273695,6900419,10.901795
49997,273695,879773,10.901794
49998,273695,94045,10.901793


In [45]:
all_result = bm25scores.merge(collection, how='left', on=['docid']).fillna(0)
all_result

,qid,docid,score,passage
0,915593,82107,25.859501,What kind of foods can you cook sous vide? Sou...
1,915593,1772930,25.299900,"Well, one of Arnold's biggest insights is what..."
2,915593,6923052,25.073299,What is a Sous Vide Cooker? We said it before ...
3,915593,8178998,23.998600,Sous-vide cooking involves cooking food in sea...
4,915593,3523599,23.721701,Actually vacuum is a mis-represented concept i...
...,...,...,...,...
49995,273695,6485471,10.901796,How long will Adderall test positive in a drug...
49996,273695,6900419,10.901795,How Long Does Marijuana (THC) Stay In Your Sys...
49997,273695,879773,10.901794,Cocaine is a fast acting drug with a short hal...
49998,273695,94045,10.901793,How Long Alcohol Stays In Your System. People ...


In [46]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
from nltk import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

def clean_text(text):

#  This function takes as input a text on which several
#  NLTK algorithms will be applied in order to preprocess it

    tokens = word_tokenize(text)
    # Remove the punctuations
    tokens = [word for word in tokens if word.isalpha()]
    # Lower the tokens
    tokens = [word.lower() for word in tokens]
    # Remove stopword
    tokens = [word for word in tokens if not word in stopwords.words("english")]
    # Lemmatize
    lemma = WordNetLemmatizer()
    tokens = [lemma.lemmatize(word, pos = "v") for word in tokens]
    tokens = [lemma.lemmatize(word, pos = "n") for word in tokens]
    return tokens

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Akaberi.mozhgan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [47]:
NDCG_scores = pd.read_csv("DL_hard/HardNDCG.tsv" , sep='\t' , names=['qid','text','score'])
NDCG_scores = NDCG_scores[['qid', 'score']]
# print(NDCG_scores)

In [48]:
index_stats = index_reader.stats()
print(index_stats)

{'total_terms': 352316036, 'documents': 8841823, 'non_empty_documents': 8841823, 'unique_terms': 2660824}


In [49]:
def bm25_similarity(query, k1=0.9, b=0.4):
    score = 0.0
    N = 8841823  # Total number of documents in the corpus
    avgdl = 32
    # avgdl = sum(len(doc) for doc in corpus) / N  # Average document length in the corpus
    analyzed_query = index_reader.analyze(query)
    # print(analyzed_query)
    
    for term in analyzed_query:
        analyzed = index_reader.analyze(term)
        # print(analyzed)
        
        # Skip term analysis:
        df, cf = index_reader.get_term_counts(term[0], analyzer=None)
        # print(f'term "{term}": df={df}, cf={cf}')
    
        # Compute the BM25 components
        idf = math.log((N - df + 0.5) / (df + 0.5))
        term_score = (idf * cf * (k1 + 1)) / (cf + k1 * (1 - b + b * (index_stats['total_terms'] / avgdl)))
        # print(term_score)
    
        score += term_score

    return score

In [161]:
# calculate Score D
results = pd.DataFrame(columns=['qid','score'])
# my_queries = dev_queries.head(10)
for i,row in hard_queries.iterrows():
  qid = row['id']
  query = row['query']
  # print(query)
  score_D = bm25_similarity(query)
    
  new_data = pd.DataFrame([[ qid , score_D ]],columns= results.columns)
  results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('DL_hard/score_D.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_2124\2707633808.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [50]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn_extra.cluster import KMedoids
import matplotlib.pyplot as plt
from decimal import Decimal
from scipy.spatial.distance import euclidean

my_list = list()

my_queries = hard_queries.head(1)
for i,row in hard_queries.iterrows():
  qid = row['id']
  retrieved_list = all_result.loc[all_result['qid'] == qid]
  documents = retrieved_list["passage"]
  # print(documents)

  # create vectorizer
  vectorizer = TfidfVectorizer(stop_words='english')

  # vectorizer the text documents
  vectorized_documents = vectorizer.fit_transform(documents)
  # print(vectorized_documents)
  matrix = vectorized_documents.toarray()
  # print(matrix)

  # reduce the dimensionality of the data using PCA
  pca = PCA(n_components=2)
  reduced_data = pca.fit_transform(vectorized_documents.toarray())

  num_clusters = 5
    
  # Cluster the documents using k-means
  kmeans = KMeans(n_clusters=num_clusters, n_init="auto",max_iter=500, random_state=42)
  kmeans.fit(vectorized_documents)

  # Cluster the data using KMedoids
  # kmedoids = KMedoids(n_clusters=num_clusters,max_iter=500, random_state=42)
  # kmedoids.fit(vectorized_documents)

  # labels = kmedoids.labels_
  # print(labels)
  # medoid_indices = kmedoids.medoid_indices_
  # print(medoid_indices)

  labels = kmeans.labels_  
  # print(labels)
  # centroids = kmeans.cluster_centers_
  # print(centroids)

# Loop over all clusters and find index of closest point to the cluster center and append to closest_pt_idx list.
  closest_pt_idx = []
  for iclust in range(kmeans.n_clusters):
      # get all points assigned to each cluster:
      cluster_pts = matrix[kmeans.labels_ == iclust]
   
      # get all indices of points assigned to this cluster:
      cluster_pts_indices = np.where(kmeans.labels_ == iclust)[0]  

      cluster_cen = kmeans.cluster_centers_[iclust]
      min_idx = np.argmin([euclidean(matrix[idx], cluster_cen) for idx in cluster_pts_indices])
      closest_pt_idx.append(cluster_pts_indices[min_idx])


  my_obj = {"qid": qid,
            "lables": labels,
            "medoids_indices": closest_pt_idx
           }

  my_list.append(my_obj)

# print(my_list)

# Store data (serialize)
with open('DL_hard/K-means/my_clusters.pickle', 'wb') as handle:
    pickle.dump(my_list, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [32]:
# Load data (deserialize)
with open('DL_hard/K-means/my_clusters.pickle', 'rb') as handle:
    clustered_data = pickle.load(handle)

# print(type(clustered_data))    
print(len(clustered_data))

50


In [162]:
# calculate Primary NQC
score_corpus = pd.read_csv("DL_hard/score_D.txt" , sep='\t' , names=['qid','score'])
# print(score_corpus)

results = pd.DataFrame(columns=['qid','score'])
# my_queries = hard_queries.head(1)
for i,row in hard_queries.iterrows():
  qid = row['id']
  retrieved_list = bm25scores.loc[bm25scores['qid'] == qid]
  # print(retrieved_list)
  nqc_score = 0
  score_D = score_corpus.loc[score_corpus['qid'] == qid, 'score'].item()
  # score_D = 1

  mean_score = mean(retrieved_list['score'])
  for index, row1 in retrieved_list.iterrows():
    score_d = row1['score']
    nqc_score += pow((score_d - mean_score),2)
    
  nqc_score = nqc_score / len(retrieved_list)
  nqc_score = math.sqrt(nqc_score)
    
  nqc_score = nqc_score / score_D
    
  new_data = pd.DataFrame([[ qid , nqc_score ]],columns= results.columns)
  results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('DL_hard/nqc_scores.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_2124\2264440728.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [33]:
# Calculate Clustering NQC
score_corpus = pd.read_csv("DL_hard/score_D.txt" , sep='\t' , names=['qid','score'])
# print(score_corpus)
score_D = score_corpus.loc[score_corpus['qid'] == qid, 'score'].item()

results = pd.DataFrame(columns=['qid','score'])
# my_queries = hard_queries.head(1)
for i,row in hard_queries.iterrows():
    qid = row['id']
    retrieved_list = all_result.loc[all_result['qid'] == qid]
    
    res = next((item for item in clustered_data if item["qid"] == qid), None)
    lables = res["lables"]
    medoids_indices = res["medoids_indices"]

    nqc_score = 0
    score_D = score_corpus.loc[score_corpus['qid'] == qid, 'score'].item()
    # score_D = 1
   
    df = pd.DataFrame({'qid': retrieved_list["qid"], 'docid': retrieved_list["docid"], 'score': retrieved_list["score"],'cluster': lables})
   
    for g, data in df.groupby('cluster'):
        medoid_doc = all_result['score'].iloc[medoids_indices[g]]
        best_docs = data['score'].iloc[:1]
        # print(best_docs)
        for j , best_doc in best_docs.items():
            nqc_score += pow((best_doc - medoid_doc),2)

    k = len(medoids_indices) * len(best_docs)
    # print(k)
    nqc_score = nqc_score / k
    nqc_score = math.sqrt(nqc_score)
        
    nqc_score = nqc_score / score_D
    # print(nqc_score)
        
    new_data = pd.DataFrame([[ qid , nqc_score ]],columns= results.columns)
    results = pd.concat([results, new_data],ignore_index=True)

# print(results)
results.to_csv('DL_hard/K-means/nqc_scores51.txt', header=None, index=None, sep='\t', mode='w')

C:\Users\Akaberi.mozhgan\AppData\Local\Temp\ipykernel_5920\3945288602.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, new_data],ignore_index=True)


In [36]:
# nqc_scores = pd.read_csv("DL_hard/nqc_scores.txt" , sep='\t' , names=['qid','score'])
nqc_scores = pd.read_csv("DL_hard/K-means/nqc_scores51.txt" , sep='\t' , names=['qid','score'])
# print(nqc_scores)

x = NDCG_scores
y = nqc_scores

sorted_x = x.sort_values(by=['qid'])
# sorted_x

sorted_y = y.sort_values(by=['qid'])
# sorted_y

In [37]:
my_data = sorted_x.merge(sorted_y, how='left', on=['qid']).fillna(0)
my_data = my_data[['score_x' , 'score_y']]
# my_data
correlation = my_data['score_x'].corr(my_data['score_y'])
print(correlation)

0.06662195486137061


In [38]:
corr = my_data.corr(method = 'pearson')
corr

,score_x,score_y
score_x,1.000000,0.066622
score_y,0.066622,1.000000


In [39]:
corr = my_data.corr(method = 'spearman')
corr

,score_x,score_y
score_x,1.000000,0.155843
score_y,0.155843,1.000000


In [40]:
corr = my_data.corr(method = 'kendall')
corr

,score_x,score_y
score_x,1.000000,0.107439
score_y,0.107439,1.000000
